# Evaluación 2: BDD

**Alumno:** Armando  
**Curso:** INF3831 - Bases de Datos

## Preguntas respondidas
- **P1** (opcional): Lógica en la BDD — Stored Procedure y Trigger
- **P3** (obligatoria): Formas Normales

---

## Esquema de la BDD

```sql
Producto(id_prod SERIAL PRIMARY KEY,
         sku VARCHAR(50),
         nombre_prod VARCHAR(100),
         descripcion_prod VARCHAR(500),
         id_cate INT,
         activo BOOLEAN,
         FOREIGN KEY(id_cate) REFERENCES Categoria(id_cate))

Categoria(id_cate SERIAL PRIMARY KEY,
          nombre_cate VARCHAR(100),
          descripcion_cate VARCHAR(100),
          activo BOOLEAN)

Lista_Precios(id_lista SERIAL PRIMARY KEY,
              inicio_vigencia DATE,
              fin_vigencia DATE,
              tipo INT)

Producto_Lista(id_prod INT,
               id_lista INT,
               precio INT,
               PRIMARY KEY (id_prod, id_lista))
```

## Setup inicial
Creamos las tablas y cargamos datos de prueba.

In [1]:
try:
    import sqlalchemy, sql
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ipython-sql", "sqlalchemy"])


In [2]:
import sqlite3, pathlib

db_path = pathlib.Path("ev2.db")
if db_path.exists():
    db_path.unlink()

conn = sqlite3.connect(str(db_path))
cur = conn.cursor()

cur.executescript("""
CREATE TABLE Categoria (
    id_cate   INTEGER PRIMARY KEY,
    nombre_cate      TEXT NOT NULL,
    descripcion_cate TEXT,
    activo           INTEGER NOT NULL  -- 1=TRUE, 0=FALSE
);

CREATE TABLE Producto (
    id_prod          INTEGER PRIMARY KEY,
    sku              TEXT,
    nombre_prod      TEXT NOT NULL,
    descripcion_prod TEXT,
    id_cate          INTEGER,
    activo           INTEGER NOT NULL,
    FOREIGN KEY (id_cate) REFERENCES Categoria(id_cate)
);

CREATE TABLE Lista_Precios (
    id_lista         INTEGER PRIMARY KEY,
    inicio_vigencia  TEXT NOT NULL,  -- almacenado como 'YYYY-MM-DD'
    fin_vigencia     TEXT NOT NULL,
    tipo             INTEGER NOT NULL  -- 0=base, 1=oferta
);

CREATE TABLE Producto_Lista (
    id_prod  INTEGER NOT NULL,
    id_lista INTEGER NOT NULL,
    precio   INTEGER NOT NULL,
    PRIMARY KEY (id_prod, id_lista)
);

-- Categorías
INSERT INTO Categoria VALUES (1, 'Electrónica',  'Dispositivos y tecnología', 1);
INSERT INTO Categoria VALUES (2, 'Ropa',          'Prendas de vestir',         1);
INSERT INTO Categoria VALUES (3, 'Hogar',         'Artículos del hogar',       0);

-- Productos (activo=1 activos, activo=0 inactivos)
INSERT INTO Producto VALUES (1, 'SKU001', 'Laptop Dell',     'Computador portátil 15"', 1, 1);
INSERT INTO Producto VALUES (2, 'SKU002', 'Samsung TV 55"',  'Televisor QLED 55"',      1, 1);
INSERT INTO Producto VALUES (3, 'SKU003', 'Polera Nike',     'Polera deportiva M',      2, 1);
INSERT INTO Producto VALUES (4, 'SKU004', 'Lámpara',         'Lámpara de escritorio',   3, 0);
INSERT INTO Producto VALUES (5, 'SKU005', 'Tablet Samsung',  'Tablet 10" 64GB',         1, 1);
INSERT INTO Producto VALUES (6, 'SKU006', 'Jean Levis 501',  'Jean clásico azul',       2, 1);

-- Listas de precios
--  id=1: lista base expirada
--  id=2: lista base vigente (2026-01-01 a 2026-12-31)
--  id=3: lista oferta vigente (2026-05-01 a 2026-05-31)
INSERT INTO Lista_Precios VALUES (1, '2025-01-01', '2025-12-31', 0);
INSERT INTO Lista_Precios VALUES (2, '2026-01-01', '2026-12-31', 0);
INSERT INTO Lista_Precios VALUES (3, '2026-05-01', '2026-05-31', 1);

-- Producto_Lista
INSERT INTO Producto_Lista VALUES (1, 1, 500000);  -- Laptop en lista base antigua
INSERT INTO Producto_Lista VALUES (1, 2, 520000);  -- Laptop en lista base vigente
INSERT INTO Producto_Lista VALUES (1, 3, 480000);  -- Laptop en oferta
INSERT INTO Producto_Lista VALUES (2, 2, 300000);  -- Samsung TV en lista base vigente
INSERT INTO Producto_Lista VALUES (3, 2,  25000);  -- Polera en lista base vigente
INSERT INTO Producto_Lista VALUES (3, 3,  20000);  -- Polera en oferta
INSERT INTO Producto_Lista VALUES (5, 2, 150000);  -- Tablet en lista base vigente
INSERT INTO Producto_Lista VALUES (6, 2,  35000);  -- Jean en lista base vigente
""")

conn.commit()
conn.close()

%load_ext sql
%config SqlMagic.feedback = False
%config SqlMagic.autopandas = True
%config SqlMagic.style = "_DEPRECATED_DEFAULT"
%sql sqlite:///ev2.db
print("Base de datos ev2.db lista.")

Base de datos ev2.db lista.


Verificamos que se cargaron correctamente las tablas:

In [3]:
%%sql
SELECT 'Categoria'    AS tabla, COUNT(*) AS filas FROM Categoria
UNION ALL
SELECT 'Producto',      COUNT(*) FROM Producto
UNION ALL
SELECT 'Lista_Precios', COUNT(*) FROM Lista_Precios
UNION ALL
SELECT 'Producto_Lista',COUNT(*) FROM Producto_Lista;

 * sqlite:///ev2.db


,tabla,filas
0,Categoria,3
1,Producto,6
2,Lista_Precios,3
3,Producto_Lista,8


---
## P1. Lógica en la BDD

### P1.1 — Stored Procedure: precio vigente por fecha y categoría

El procedimiento almacenado recibe una **fecha** y un **id_cate**, y retorna el precio vigente de todos los productos activos de esa categoría en esa fecha.  
Si el producto está en una lista de oferta vigente, se usa ese precio; de lo contrario se usa el precio base vigente.

#### Código PostgreSQL

```sql
CREATE OR REPLACE FUNCTION obtener_precios_vigentes(
    p_fecha  DATE,
    p_id_cate INT
)
RETURNS TABLE (
    id_prod      INT,
    nombre_prod  VARCHAR,
    precio       INT,
    tipo_precio  TEXT
)
LANGUAGE plpgsql
AS $$
BEGIN
    RETURN QUERY
    WITH oferta AS (
        -- productos que tienen una lista de oferta vigente en la fecha dada
        SELECT pl.id_prod, pl.precio
        FROM Producto_Lista pl
        JOIN Lista_Precios lp ON pl.id_lista = lp.id_lista
        WHERE lp.tipo = 1
          AND p_fecha BETWEEN lp.inicio_vigencia AND lp.fin_vigencia
    ),
    base AS (
        -- precio base vigente para cada producto
        SELECT pl.id_prod, pl.precio
        FROM Producto_Lista pl
        JOIN Lista_Precios lp ON pl.id_lista = lp.id_lista
        WHERE lp.tipo = 0
          AND p_fecha BETWEEN lp.inicio_vigencia AND lp.fin_vigencia
    )
    SELECT
        p.id_prod,
        p.nombre_prod,
        COALESCE(o.precio, b.precio) AS precio,
        CASE WHEN o.precio IS NOT NULL THEN 'oferta' ELSE 'base' END AS tipo_precio
    FROM Producto p
    LEFT JOIN oferta o ON p.id_prod = o.id_prod
    LEFT JOIN base   b ON p.id_prod = b.id_prod
    WHERE p.activo = TRUE
      AND p.id_cate = p_id_cate
      AND COALESCE(o.precio, b.precio) IS NOT NULL;
END;
$$;

-- Ejemplo de uso:
SELECT * FROM obtener_precios_vigentes('2026-05-15', 1);
```

#### Demostración en SQLite

SQLite no soporta stored procedures nativamente, por lo que demostramos la lógica equivalente con una consulta SQL y la encapsulamos en Python.

In [4]:
%%sql
-- Equivalente a: obtener_precios_vigentes('2026-05-15', 1)
-- Productos activos de Electrónica (id_cate=1) con precio vigente al 2026-05-15
WITH oferta AS (
    SELECT pl.id_prod, pl.precio
    FROM Producto_Lista pl
    JOIN Lista_Precios lp ON pl.id_lista = lp.id_lista
    WHERE lp.tipo = 1
      AND '2026-05-15' BETWEEN lp.inicio_vigencia AND lp.fin_vigencia
),
base AS (
    SELECT pl.id_prod, pl.precio
    FROM Producto_Lista pl
    JOIN Lista_Precios lp ON pl.id_lista = lp.id_lista
    WHERE lp.tipo = 0
      AND '2026-05-15' BETWEEN lp.inicio_vigencia AND lp.fin_vigencia
)
SELECT
    p.id_prod,
    p.nombre_prod,
    COALESCE(o.precio, b.precio) AS precio,
    CASE WHEN o.precio IS NOT NULL THEN 'oferta' ELSE 'base' END AS tipo_precio
FROM Producto p
LEFT JOIN oferta o ON p.id_prod = o.id_prod
LEFT JOIN base   b ON p.id_prod = b.id_prod
WHERE p.activo = 1
  AND p.id_cate = 1
  AND COALESCE(o.precio, b.precio) IS NOT NULL;

 * sqlite:///ev2.db


,id_prod,nombre_prod,precio,tipo_precio
0,1,Laptop Dell,480000,oferta
1,2,"Samsung TV 55""",300000,base
2,5,Tablet Samsung,150000,base


In [5]:
import sqlite3, pandas as pd

QUERY = """
WITH oferta AS (
    SELECT pl.id_prod, pl.precio
    FROM Producto_Lista pl
    JOIN Lista_Precios lp ON pl.id_lista = lp.id_lista
    WHERE lp.tipo = 1
      AND :fecha BETWEEN lp.inicio_vigencia AND lp.fin_vigencia
),
base AS (
    SELECT pl.id_prod, pl.precio
    FROM Producto_Lista pl
    JOIN Lista_Precios lp ON pl.id_lista = lp.id_lista
    WHERE lp.tipo = 0
      AND :fecha BETWEEN lp.inicio_vigencia AND lp.fin_vigencia
)
SELECT
    p.id_prod,
    p.nombre_prod,
    COALESCE(o.precio, b.precio) AS precio,
    CASE WHEN o.precio IS NOT NULL THEN 'oferta' ELSE 'base' END AS tipo_precio
FROM Producto p
LEFT JOIN oferta o ON p.id_prod = o.id_prod
LEFT JOIN base   b ON p.id_prod = b.id_prod
WHERE p.activo = 1
  AND p.id_cate = :id_cate
  AND COALESCE(o.precio, b.precio) IS NOT NULL;
"""

def obtener_precios_vigentes(fecha: str, id_cate: int) -> pd.DataFrame:
    """Simula el stored procedure: retorna precios vigentes por fecha y categoría."""
    with sqlite3.connect("ev2.db") as conn:
        return pd.read_sql_query(QUERY, conn, params={"fecha": fecha, "id_cate": id_cate})

print("=== Electrónica (id_cate=1) al 2026-05-15 — oferta activa ===")
display(obtener_precios_vigentes("2026-05-15", 1))

print("\n=== Ropa (id_cate=2) al 2026-05-15 — oferta activa ===")
display(obtener_precios_vigentes("2026-05-15", 2))

print("\n=== Electrónica (id_cate=1) al 2026-07-01 — sin oferta activa ===")
display(obtener_precios_vigentes("2026-07-01", 1))

=== Electrónica (id_cate=1) al 2026-05-15 — oferta activa ===


,id_prod,nombre_prod,precio,tipo_precio
0,1,Laptop Dell,480000,oferta
1,2,"Samsung TV 55""",300000,base
2,5,Tablet Samsung,150000,base



=== Ropa (id_cate=2) al 2026-05-15 — oferta activa ===


,id_prod,nombre_prod,precio,tipo_precio
0,3,Polera Nike,20000,oferta
1,6,Jean Levis 501,35000,base



=== Electrónica (id_cate=1) al 2026-07-01 — sin oferta activa ===


,id_prod,nombre_prod,precio,tipo_precio
0,1,Laptop Dell,520000,base
1,2,"Samsung TV 55""",300000,base
2,5,Tablet Samsung,150000,base


---
### P1.2 — Trigger: crear tabla `Productos_Sin_Precio_YYYYMMDD`

El trigger se activa **después de insertar** un registro en `Lista_Precios`.  
Crea la tabla `Productos_Sin_Precio_YYYYMMDD` (con la fecha actual) e inserta todos los **productos activos** que **no pertenecen** a ninguna lista de ofertas (`tipo = 1`).

La tabla creada contiene los campos: `id_prod`, `sku`, `nombre_prod`, `id_cate`.

#### Código PostgreSQL

```sql
CREATE OR REPLACE FUNCTION fn_productos_sin_precio()
RETURNS TRIGGER
LANGUAGE plpgsql
AS $$
DECLARE
    v_tabla TEXT;
BEGIN
    -- Nombre dinámico con fecha actual
    v_tabla := 'Productos_Sin_Precio_' || TO_CHAR(CURRENT_DATE, 'YYYYMMDD');

    -- Crear la tabla si no existe
    EXECUTE format('
        CREATE TABLE IF NOT EXISTS %I (
            id_prod     INT,
            sku         VARCHAR(50),
            nombre_prod VARCHAR(100),
            id_cate     INT
        )', v_tabla);

    -- Insertar productos activos sin lista de oferta
    EXECUTE format('
        INSERT INTO %I (id_prod, sku, nombre_prod, id_cate)
        SELECT p.id_prod, p.sku, p.nombre_prod, p.id_cate
        FROM Producto p
        WHERE p.activo = TRUE
          AND p.id_prod NOT IN (
              SELECT pl.id_prod
              FROM Producto_Lista pl
              JOIN Lista_Precios lp ON pl.id_lista = lp.id_lista
              WHERE lp.tipo = 1
          )', v_tabla);

    RETURN NEW;
END;
$$;

CREATE TRIGGER trg_lista_precios_insert
AFTER INSERT ON Lista_Precios
FOR EACH ROW
EXECUTE FUNCTION fn_productos_sin_precio();
```

#### Demostración en SQLite

Simulamos la lógica del trigger con Python: al insertar en `Lista_Precios`, ejecutamos la misma lógica manualmente.

In [6]:
import sqlite3, datetime, pandas as pd

def simular_trigger_lista_precios(id_lista, inicio, fin, tipo):
    """Simula el trigger AFTER INSERT ON Lista_Precios."""
    fecha_hoy = datetime.date.today().strftime("%Y%m%d")
    nombre_tabla = f"Productos_Sin_Precio_{fecha_hoy}"

    with sqlite3.connect("ev2.db") as conn:
        cur = conn.cursor()

        # Insertar en Lista_Precios
        cur.execute(
            "INSERT INTO Lista_Precios VALUES (?, ?, ?, ?)",
            (id_lista, inicio, fin, tipo)
        )

        # Lógica del trigger
        cur.execute(f"""
            CREATE TABLE IF NOT EXISTS \"{nombre_tabla}\" (
                id_prod     INTEGER,
                sku         TEXT,
                nombre_prod TEXT,
                id_cate     INTEGER
            )
        """)

        cur.execute(f"""
            INSERT INTO \"{nombre_tabla}\" (id_prod, sku, nombre_prod, id_cate)
            SELECT p.id_prod, p.sku, p.nombre_prod, p.id_cate
            FROM Producto p
            WHERE p.activo = 1
              AND p.id_prod NOT IN (
                  SELECT pl.id_prod
                  FROM Producto_Lista pl
                  JOIN Lista_Precios lp ON pl.id_lista = lp.id_lista
                  WHERE lp.tipo = 1
              )
        """)
        conn.commit()

        df = pd.read_sql_query(f'SELECT * FROM "{nombre_tabla}"', conn)

    print(f"Trigger ejecutado → tabla '{nombre_tabla}' creada con {len(df)} registros:")
    return df

# Simulamos insertar una nueva lista de precios (id=4, base, vigencia futura)
resultado = simular_trigger_lista_precios(4, '2026-06-01', '2026-06-30', 0)
display(resultado)

Trigger ejecutado → tabla 'Productos_Sin_Precio_20260518' creada con 3 registros:


,id_prod,sku,nombre_prod,id_cate
0,2,SKU002,"Samsung TV 55""",1
1,5,SKU005,Tablet Samsung,1
2,6,SKU006,Jean Levis 501,2


---
## P3. Formas Normales

Relación analizada:

**Libros(ISBN, Titulo, AutorID, NombreAutor, CiudadAutor)**

Datos del enunciado:
- `ISBN` es la llave primaria.
- Un autor puede escribir múltiples libros.
- Cada autor vive en una única ciudad.

---

### P3.1 — Dependencias Funcionales

| # | Dependencia funcional | Justificación |
|---|---|---|
| 1 | ISBN → Titulo | Cada ISBN identifica un único libro con un título. |
| 2 | ISBN → AutorID | Cada ISBN corresponde a un único autor. |
| 3 | ISBN → NombreAutor | Se deduce transitivamente vía AutorID. |
| 4 | ISBN → CiudadAutor | Se deduce transitivamente vía AutorID. |
| 5 | AutorID → NombreAutor | Cada autor tiene un único nombre. |
| 6 | AutorID → CiudadAutor | Cada autor vive en una única ciudad. |

> **Nota:** No existe dependencia NombreAutor → CiudadAutor porque podría haber dos autores con el mismo nombre que vivan en ciudades distintas. La dependencia se da sobre `AutorID`.

---

### P3.2 — ¿Está en Tercera Forma Normal (3NF)?

**No**, la tabla **no está en 3NF**.

Para estar en 3NF, toda dependencia funcional no trivial `X → A` debe cumplir al menos una de las siguientes condiciones:
1. `X` es superclave.
2. `A` es un atributo primo (parte de alguna llave candidata).

En esta relación se tienen las dependencias transitivas:

```
ISBN → AutorID → NombreAutor
ISBN → AutorID → CiudadAutor
```

**Problema:** `AutorID → NombreAutor` y `AutorID → CiudadAutor` son dependencias donde:
- `AutorID` **no es superclave** (no determina `ISBN` ni `Titulo`).
- `NombreAutor` y `CiudadAutor` **no son atributos primos** (no forman parte de ninguna llave candidata).

Por lo tanto, hay **dependencias transitivas** que violan la 3NF.

---

### P3.3 — Descomposición en 3NF y BCNF

Para eliminar las dependencias transitivas, descomponemos la relación en dos tablas:

#### Esquema descompuesto

```
Libros (ISBN, Titulo, AutorID)
    PK: ISBN
    FK: AutorID → Autores(AutorID)

Autores (AutorID, NombreAutor, CiudadAutor)
    PK: AutorID
```

#### ¿Por qué cumple con 3NF y BCNF?

| Tabla | Dependencias funcionales | Cumple BCNF |
|---|---|---|
| **Libros** | `ISBN → {Titulo, AutorID}` | `ISBN` es superclave → toda DF parte de la llave |
| **Autores** | `AutorID → {NombreAutor, CiudadAutor}` | `AutorID` es superclave → toda DF parte de la llave |

En **BCNF**, para toda dependencia funcional no trivial `X → Y`, `X` debe ser superclave. Ambas tablas cumplen esta condición. Por lo tanto, la descomposición cumple tanto **3NF** como **BCNF**.

#### Propiedades de la descomposición

- **Lossless-join**: la descomposición es sin pérdida porque `AutorID` (atributo común) es llave primaria en `Autores`, garantizando que el JOIN recupera la relación original.
- **Dependency-preserving**: todas las dependencias funcionales originales se preservan en las tablas resultantes.

In [7]:
%%sql
-- Ilustración de la descomposición con datos de ejemplo

DROP TABLE IF EXISTS Libros;
DROP TABLE IF EXISTS Autores;

CREATE TABLE Autores (
    AutorID     INTEGER PRIMARY KEY,
    NombreAutor TEXT NOT NULL,
    CiudadAutor TEXT NOT NULL
);

CREATE TABLE Libros (
    ISBN    TEXT PRIMARY KEY,
    Titulo  TEXT NOT NULL,
    AutorID INTEGER NOT NULL,
    FOREIGN KEY (AutorID) REFERENCES Autores(AutorID)
);

INSERT OR REPLACE INTO Autores VALUES (1, 'Gabriel García Márquez', 'Bogotá');
INSERT OR REPLACE INTO Autores VALUES (2, 'Isabel Allende',         'Lima');
INSERT OR REPLACE INTO Autores VALUES (3, 'Jorge Luis Borges',      'Buenos Aires');

INSERT OR REPLACE INTO Libros VALUES ('978-0-06-088328-7', 'Cien años de soledad',  1);
INSERT OR REPLACE INTO Libros VALUES ('978-0-06-093208-9', 'El amor en los tiempos del cólera', 1);
INSERT OR REPLACE INTO Libros VALUES ('978-1-5011-3588-8', 'La casa de los espíritus', 2);
INSERT OR REPLACE INTO Libros VALUES ('978-0-14-243723-0', 'Ficciones',              3);

SELECT L.ISBN, L.Titulo, A.AutorID, A.NombreAutor, A.CiudadAutor
FROM Libros L
JOIN Autores A ON L.AutorID = A.AutorID;

 * sqlite:///ev2.db


,ISBN,Titulo,AutorID,NombreAutor,CiudadAutor
0,978-0-06-088328-7,Cien años de soledad,1,Gabriel García Márquez,Bogotá
1,978-0-06-093208-9,El amor en los tiempos del cólera,1,Gabriel García Márquez,Bogotá
2,978-1-5011-3588-8,La casa de los espíritus,2,Isabel Allende,Lima
3,978-0-14-243723-0,Ficciones,3,Jorge Luis Borges,Buenos Aires


In [8]:
# Compilar reporte LaTeX a PDF
import subprocess

result = subprocess.run(
    ["pdflatex", "-interaction=nonstopmode", "evaluacion2_report.tex"],
    capture_output=True
)
if result.returncode == 0:
    print("PDF generado: evaluacion2_report.pdf")
else:
    print(result.stdout.decode("latin-1", errors="replace")[-2000:])


PDF generado: evaluacion2_report.pdf
